In [ ]:
# @title Unzip {"vertical-output":true}
#
# 📝 REVISION HISTORY
# v2.4 - Removed auto-restart/disk guard. Focus on raw extraction & resume capability.
# v2.3 - Replaced cleaning with automatic Runtime Restart.

import os
import zipfile
import shutil
import time
import sys
from google.colab import drive, output
from tqdm.notebook import tqdm

def mount_gd():
    if not os.path.exists('/content/drive'):
        print("📄 Mounting Google Drive...")
        drive.mount('/content/drive')
    else:
        print("✅ Google Drive already mounted.")

def check_disk_usage():
    total, used, free = shutil.disk_usage("/")
    return (used / total) * 100

def browse_path(start_path, prompt_message, select_item_type=None):
    current_path = start_path
    while True:
        output.clear()
        print(f"\n--- ✨ {prompt_message} ✨ ---")
        print(f"📍 Current Path: {current_path}\n")
        try:
            items = sorted(os.listdir(current_path))
        except Exception as e:
            print(f"❌ Error: {e}"); time.sleep(2); current_path = os.path.dirname(current_path); continue
        dirs = [i for i in items if os.path.isdir(os.path.join(current_path, i))]
        files = [i for i in items if not os.path.isdir(os.path.join(current_path, i))]
        print("📁 Directories:")
        for i, d in enumerate(dirs): print(f"  [{i+1}] {d}")
        print("\n📄 Files:")
        for i, f in enumerate(files): print(f"  [{len(dirs)+i+1}] {f}")
        print("\n🛠️ Options: [s] Select Current [b] Back [n] New Folder [q] Quit")
        try:
            choice = input("👉 Choice: ").strip().lower()
        except EOFError: return None
        if choice == 'q': return None
        elif choice == 'b': current_path = os.path.dirname(current_path)
        elif choice == 'n':
            name = input("🆕 New folder name: ").strip()
            if name: os.makedirs(os.path.join(current_path, name), exist_ok=True)
        elif choice == 's':
            if select_item_type == 'zip':
                zips = [f for f in files if f.lower().endswith('.zip')]
                if not zips: print("⚠️ No ZIPs."); time.sleep(2); continue
                return os.path.join(current_path, zips[0])
            return current_path
        else:
            try:
                idx = int(choice) - 1
                if idx < len(dirs): current_path = os.path.join(current_path, dirs[idx])
                elif idx < (len(dirs)+len(files)):
                    f = files[idx-len(dirs)]
                    if select_item_type == 'zip' and f.lower().endswith('.zip'): return os.path.join(current_path, f)
            except: pass

def extract_zip(zip_path, dest_path):
    try:
        with zipfile.ZipFile(zip_path, 'r') as zf:
            infolist = zf.infolist()
            total_size = sum(file.file_size for file in infolist)
            print(f"\n📦 Processing ZIP: {os.path.basename(zip_path)}")

            with tqdm(total=total_size, unit='B', unit_scale=True, desc="🔓 Extracting", colour='green') as pbar:
                for i, member in enumerate(infolist):
                    target_path = os.path.join(dest_path, member.filename)
                    if member.is_dir():
                        os.makedirs(target_path, exist_ok=True)
                        pbar.update(member.file_size); continue

                    # Skip Logic for Resume (Crucial for manual restarts)
                    if os.path.exists(target_path) and os.path.getsize(target_path) == member.file_size:
                        pbar.update(member.file_size); continue

                    # Visual Feedback Only (No Auto-Restart)
                    if i % 10 == 0:
                        usage = check_disk_usage()
                        pbar.set_postfix_str(f"🖥️ Disk: {usage:.1f}%")

                    os.makedirs(os.path.dirname(target_path), exist_ok=True)
                    with zf.open(member) as source, open(target_path, "wb") as target:
                        while True:
                            chunk = source.read(2 * 1024 * 1024)
                            if not chunk: break
                            target.write(chunk)
                            pbar.update(len(chunk))
                        target.flush(); os.fsync(target.fileno())

        print(f"\n✅ Extraction Complete!")
    except Exception as e:
        print(f"\n❌ Stopped: {e}")
        print("💡 Progress saved. You can restart the runtime and run the cell again to resume.")

mount_gd()
drive_root = '/content/drive/My Drive'
selected_zip = browse_path(drive_root, "Select ZIP", 'zip')
if selected_zip:
    dest_dir = browse_path(drive_root, "Select Dest", 'directory')
    if dest_dir:
        mode = ""
        while mode not in ['1', '2']:
            print("\n🤔 Mode:\n1️⃣ [1] Extract (Direct Streaming)\n2️⃣ [2] Split ZIP")
            mode = input("👉 Choice (1 or 2): ").strip()
        if mode == '1': extract_zip(selected_zip, dest_dir)
        print(f"\n🏁 Finished: {time.ctime()}")

In [ ]:
# @title 7zip Mark III (FUSE-Aware Freeze/Thaw Engine + Live ETA) {"vertical-output":true}
#
# 📝 REVISION HISTORY
# v4.2 (Mark III) - Fixed Drive Mount error with force_remount logic.
# v4.1 (Mark III) - Added live ETA, percentage tracking, and active file display via tqdm.
# v4.0 (Mark III) - Added background thread to monitor FUSE cache overflow.

import os
import time
import sys
import subprocess
import threading
import signal
import shutil
import re
from tqdm.notebook import tqdm
from google.colab import drive, output

def mount_gd():
    try:
        if not os.path.exists('/content/drive'):
            print("📄 Mounting Google Drive...")
            drive.mount('/content/drive', force_remount=True)
        else:
            print("✅ Google Drive already mounted.")
    except Exception as e:
        print(f"❌ Mount failed: {e}")
        print("💡 Tip: Try clicking the 'Files' icon on the left and selecting 'Mount Drive' manually if this persists.")

def check_disk_usage():
    total, used, free = shutil.disk_usage("/")
    return (used / total) * 100

def browse_path(start_path, prompt_message, select_item_type=None):
    current_path = start_path
    while True:
        output.clear()
        print(f"\n--- ✨ {prompt_message} ✨ ---")
        print(f"📍 Current Path: {current_path}\n")
        try:
            if not os.path.exists(current_path):
                print(f"⚠️ Path not found: {current_path}")
                time.sleep(2)
                current_path = "/content/drive/My Drive"
                continue
            items = sorted(os.listdir(current_path))
        except Exception as e:
            print(f"❌ Error: {e}"); time.sleep(2); current_path = os.path.dirname(current_path); continue

        dirs = [i for i in items if os.path.isdir(os.path.join(current_path, i))]
        files = [i for i in items if not os.path.isdir(os.path.join(current_path, i))]

        print("📁 Directories:")
        for i, d in enumerate(dirs): print(f"  [{i+1}] {d}")
        print("\n📄 Files:")
        for i, f in enumerate(files): print(f"  [{len(dirs)+i+1}] {f}")
        print("\n🛠️ Options: [s] Select Current [b] Back [n] New Folder [q] Quit")

        try:
            choice = input("👉 Choice: ").strip().lower()
        except EOFError: return None

        if choice == 'q': return None
        elif choice == 'b': current_path = os.path.dirname(current_path)
        elif choice == 'n':
            name = input("🆕 New folder name: ").strip()
            if name: os.makedirs(os.path.join(current_path, name), exist_ok=True)
        elif choice == 's':
            if select_item_type == 'zip':
                zips = [f for f in files if f.lower().endswith('.zip')]
                if not zips: print("⚠️ No ZIPs."); time.sleep(2); continue
                return os.path.join(current_path, zips[0])
            return current_path
        else:
            try:
                idx = int(choice) - 1
                if idx < len(dirs): current_path = os.path.join(current_path, dirs[idx])
                elif idx < (len(dirs)+len(files)):
                    f = files[idx-len(dirs)]
                    if select_item_type == 'zip' and f.lower().endswith('.zip'): return os.path.join(current_path, f)
            except: pass

def disk_monitor(process):
    while process.poll() is None:
        usage = check_disk_usage()
        if usage > 85.0:
            tqdm.write(f"\n⚠️ VM Disk at {usage:.1f}%! Pausing 7-Zip to let Google Drive sync...")
            process.send_signal(signal.SIGSTOP)
            while check_disk_usage() > 65.0:
                time.sleep(5)
            tqdm.write(f"✅ Sync complete. VM Disk at {check_disk_usage():.1f}%. Resuming 7-Zip...\n")
            process.send_signal(signal.SIGCONT)
        time.sleep(2)

def extract_with_7z(zip_path, dest_path):
    print("⚙️ Checking/Installing 7-Zip engine...")
    os.system("apt-get update > /dev/null 2>&1")
    os.system("apt-get install p7zip-full -y > /dev/null 2>&1")

    print(f"\n📦 Processing ZIP natively: {os.path.basename(zip_path)}")
    print("🚀 Using 7-Zip with FUSE Overflow Protection")
    print("-" * 50)

    cmd = ['7z', 'x', zip_path, f'-o{dest_path}', '-aos', '-bsp1']

    try:
        process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        monitor_thread = threading.Thread(target=disk_monitor, args=(process,))
        monitor_thread.daemon = True
        monitor_thread.start()

        pbar = tqdm(total=100, desc="🔓 Extracting", unit='%', colour='green')
        current_pct = 0
        buffer = b""

        while True:
            char = process.stdout.read(1)
            if not char and process.poll() is not None: break
            if char == b'\r' or char == b'\n':
                line = buffer.decode('utf-8', errors='ignore').strip()
                buffer = b""
                if not line: continue
                pct_match = re.search(r'(\d+)%', line)
                if pct_match:
                    pct = int(pct_match.group(1))
                    if pct > current_pct and pct <= 100:
                        pbar.update(pct - current_pct)
                        current_pct = pct
                if "- " in line and "%" in line:
                    filename = line.split("- ", 1)[-1].strip()
                    if len(filename) > 40: filename = "..." + filename[-37:]
                    pbar.set_postfix_str(f"📄 {filename}")
                elif "Extracting" in line:
                    filename = line.split("Extracting", 1)[-1].strip()
                    if len(filename) > 40: filename = "..." + filename[-37:]
                    pbar.set_postfix_str(f"📄 {filename}")
            else:
                buffer += char

        pbar.close()
        process.wait()
        if process.returncode == 0:
            print(f"\n✅ Extraction Complete!")
        else:
            print(f"\n⚠️ Extraction stopped or threw a warning (Code: {process.returncode})")
    except Exception as e:
        print(f"\n❌ Error executing 7-Zip: {e}")

mount_gd()
drive_root = '/content/drive/My Drive'
selected_zip = browse_path(drive_root, "Select ZIP", 'zip')

if selected_zip:
    dest_dir = browse_path(drive_root, "Select Dest", 'directory')
    if dest_dir:
        extract_with_7z(selected_zip, dest_dir)
        print(f"\n🏁 Finished: {time.ctime()}")

In [ ]:
# @title clear rclone.config (from gdrive) {"vertical-output":true}

import os
import shutil

# 1. Delete the temporary config
local_conf = os.path.expanduser("~/.config/rclone/rclone.conf")
if os.path.exists(local_conf):
    os.remove(local_conf)
    print("🗑️ Deleted local Rclone config.")

# 2. Delete the backup on Google Drive
backup_conf = '/content/drive/MyDrive/rclone_backup.conf'
if os.path.exists(backup_conf):
    os.remove(backup_conf)
    print("🗑️ Deleted corrupted backup from Google Drive.")

print("✅ System wiped. You can now re-run the Mark IV script and carefully paste the [gdrive] block again.")

✅ System wiped. You can now re-run the Mark IV script and carefully paste the [gdrive] block again.
